In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
import cudf
import gc

Enabled rmm statistics


In [2]:
import numpy as np
import pandas as pd
dummy_i64 = pd.Series([0], dtype="int64")   # 1-row int64 Series
cudf.Series(dummy_i64)                      # triggers int64 kernels & pool alloc
dummy_str = pd.Series(['warm-up'], dtype='object')   # <— 1-row object column
cudf.Series(dummy_str)                               # triggers string-kernels

0    warm-up
dtype: object

# Exploratory Data Analysis

In this notebook we will look into the speedtest data gathered from ookla and draw some useful insights from the dataset. Additional dataset would also be utilized to make some meaningful insights into the appropriately combined wholistic data

## File Exploration

In the following block we see all the files available for this kernel. The relevant libraries are also imported to process the data extracted from the dataset.

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import regex as re # For String searches
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2020_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2020_quarter_01.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_01.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_03.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2020_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2021_quarter_02.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2020_quarter_03.csv
/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/mobile_year_2022_quarter_01.csv
/home/dias-benchmarks/notebooks/gk

In [4]:
%%time
### cell 0 ###

data = pd.read_csv('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2021_quarter_03.csv')
data.head()

CPU times: user 5.81 ms, sys: 261 μs, total: 6.07 ms
Wall time: 4.44 ms


,Name,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency
0,Afghanistan,897,"2,312","9,003","3,983","4,823",70,"25,067,407",217,231,33
1,Åland Islands,284,580,"1,341","58,693","82,767",11,0,27,47,215
2,Albania,"7,215","38,905","104,752","17,354","25,803",20,"3,153,731",100,127,159
3,Algeria,"16,056","70,929","413,666","1,508","9,057",43,"32,854,159",234,211,68
4,American Samoa,80,202,900,"14,345","33,078",18,"64,051",119,104,171


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               236 non-null    object
 1   Number of Records  236 non-null    object
 2   Devices            236 non-null    object
 3   Tests              236 non-null    object
 4   Avg. Avg U Kbps    236 non-null    object
 5   Avg. Avg D Kbps    236 non-null    object
 6   Avg Lat Ms         236 non-null    int64 
 7   Avg. Pop2005       236 non-null    object
 8   Rank Upload        236 non-null    int64 
 9   Rank Download      236 non-null    int64 
 10  Rank Latency       236 non-null    int64 
dtypes: int64(4), object(7)
memory usage: 20.4+ KB


In [6]:
cudf_data = cudf.DataFrame() 

In [7]:
%%time
## Transfer_post 0 ##
cudf_data['Name'] = data['Name']

CPU times: user 2.64 ms, sys: 0 ns, total: 2.64 ms
Wall time: 1.46 ms


In [8]:
%%time
## Transfer_post 0 ##
cudf_data['Number of Records'] = data['Number of Records']

CPU times: user 1.55 ms, sys: 222 μs, total: 1.77 ms
Wall time: 1.1 ms


In [9]:
%%time
## Transfer_post 0 ##
cudf_data['Devices'] = data['Devices']

CPU times: user 1.61 ms, sys: 233 μs, total: 1.85 ms
Wall time: 1.04 ms


In [10]:
%%time
## Transfer_post 0 ##
cudf_data['Tests'] = data['Tests']

CPU times: user 1.68 ms, sys: 245 μs, total: 1.93 ms
Wall time: 1.16 ms


In [11]:
%%time
## Transfer_post 0 ##
cudf_data['Avg. Avg U Kbps'] = data['Avg. Avg U Kbps']

CPU times: user 955 μs, sys: 139 μs, total: 1.09 ms
Wall time: 901 μs


In [12]:
%%time
## Transfer_post 0 ##
cudf_data['Avg. Avg D Kbps'] = data['Avg. Avg D Kbps']

CPU times: user 2.06 ms, sys: 0 ns, total: 2.06 ms
Wall time: 1.32 ms


In [13]:
%%time
## Transfer_post 0 ##
cudf_data['Avg Lat Ms'] = data['Avg Lat Ms']

CPU times: user 871 μs, sys: 127 μs, total: 998 μs
Wall time: 813 μs


In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               236 non-null    object
 1   Number of Records  236 non-null    object
 2   Devices            236 non-null    object
 3   Tests              236 non-null    object
 4   Avg. Avg U Kbps    236 non-null    object
 5   Avg. Avg D Kbps    236 non-null    object
 6   Avg Lat Ms         236 non-null    int64 
 7   Avg. Pop2005       236 non-null    object
 8   Rank Upload        236 non-null    int64 
 9   Rank Download      236 non-null    int64 
 10  Rank Latency       236 non-null    int64 
dtypes: int64(4), object(7)
memory usage: 20.4+ KB


In [15]:
%%time
## Transfer_post 0 ##
cudf_data['Avg. Pop2005'] = data['Avg. Pop2005']

CPU times: user 1.13 ms, sys: 0 ns, total: 1.13 ms
Wall time: 868 μs


In [16]:
%%time
## Transfer_post 0 ##
cudf_data['Rank Upload'] = data['Rank Upload']

CPU times: user 1.39 ms, sys: 0 ns, total: 1.39 ms
Wall time: 915 μs


In [17]:
%%time
## Transfer_post 0 ##
cudf_data['Rank Download'] = data['Rank Download']

CPU times: user 517 μs, sys: 76 μs, total: 593 μs
Wall time: 545 μs


In [18]:
%%time
## Transfer_post 0 ##
cudf_data['Rank Latency'] = data['Rank Latency']

CPU times: user 1.29 ms, sys: 0 ns, total: 1.29 ms
Wall time: 858 μs


In [19]:
del cudf_data; gc.collect(); cudf_data = cudf.DataFrame() 

In [20]:
%%time
## Transfer_pre 1 ##
cudf_data['Name'] = data['Name']

CPU times: user 1.55 ms, sys: 0 ns, total: 1.55 ms
Wall time: 1.26 ms


In [21]:
%%time
## Transfer_pre 1 ##
cudf_data['Number of Records'] = data['Number of Records']

CPU times: user 1.65 ms, sys: 128 μs, total: 1.78 ms
Wall time: 1.06 ms


In [22]:
%%time
## Transfer_pre 1 ##
cudf_data['Devices'] = data['Devices']

CPU times: user 0 ns, sys: 1.12 ms, total: 1.12 ms
Wall time: 851 μs


In [23]:
%%time
## Transfer_pre 1 ##
cudf_data['Tests'] = data['Tests']

CPU times: user 1.42 ms, sys: 203 μs, total: 1.62 ms
Wall time: 981 μs


In [24]:
%%time
## Transfer_pre 1 ##
cudf_data['Avg. Avg U Kbps'] = data['Avg. Avg U Kbps']

CPU times: user 1.32 ms, sys: 188 μs, total: 1.51 ms
Wall time: 948 μs


In [25]:
%%time
## Transfer_pre 1 ##
cudf_data['Avg. Avg D Kbps'] = data['Avg. Avg D Kbps']

CPU times: user 941 μs, sys: 134 μs, total: 1.08 ms
Wall time: 821 μs


In [26]:
%%time
## Transfer_pre 1 ##
cudf_data['Avg Lat Ms'] = data['Avg Lat Ms']

CPU times: user 1.29 ms, sys: 185 μs, total: 1.48 ms
Wall time: 1.04 ms


In [27]:
%%time
## Transfer_pre 1 ##
cudf_data['Avg. Pop2005'] = data['Avg. Pop2005']

CPU times: user 1.08 ms, sys: 154 μs, total: 1.23 ms
Wall time: 951 μs


In [28]:
%%time
## Transfer_pre 1 ##
cudf_data['Rank Upload'] = data['Rank Upload']

CPU times: user 967 μs, sys: 139 μs, total: 1.11 ms
Wall time: 735 μs


In [29]:
%%time
## Transfer_pre 1 ##
cudf_data['Rank Download'] = data['Rank Download']

CPU times: user 599 μs, sys: 0 ns, total: 599 μs
Wall time: 557 μs


In [30]:
%%time
## Transfer_pre 1 ##
cudf_data['Rank Latency'] = data['Rank Latency']

CPU times: user 589 μs, sys: 0 ns, total: 589 μs
Wall time: 539 μs


In [31]:
%%time
### cell 1 ###

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               236 non-null    object
 1   Number of Records  236 non-null    object
 2   Devices            236 non-null    object
 3   Tests              236 non-null    object
 4   Avg. Avg U Kbps    236 non-null    object
 5   Avg. Avg D Kbps    236 non-null    object
 6   Avg Lat Ms         236 non-null    int64 
 7   Avg. Pop2005       236 non-null    object
 8   Rank Upload        236 non-null    int64 
 9   Rank Download      236 non-null    int64 
 10  Rank Latency       236 non-null    int64 
dtypes: int64(4), object(7)
memory usage: 20.4+ KB
CPU times: user 0 ns, sys: 3.88 ms, total: 3.88 ms
Wall time: 3.66 ms


In [32]:
del cudf_data; gc.collect(); cudf_data = cudf.DataFrame() 

In [33]:
%%time
## Transfer_pre 2 ##
cudf_data['Name'] = data['Name']

CPU times: user 1.41 ms, sys: 0 ns, total: 1.41 ms
Wall time: 1.13 ms


In [34]:
%%time
## Transfer_pre 2 ##
cudf_data['Number of Records'] = data['Number of Records']

CPU times: user 1.27 ms, sys: 179 μs, total: 1.45 ms
Wall time: 937 μs


In [35]:
%%time
## Transfer_pre 2 ##
cudf_data['Devices'] = data['Devices']

CPU times: user 1.18 ms, sys: 0 ns, total: 1.18 ms
Wall time: 869 μs


In [36]:
%%time
## Transfer_pre 2 ##
cudf_data['Tests'] = data['Tests']

CPU times: user 1.49 ms, sys: 209 μs, total: 1.7 ms
Wall time: 1.05 ms


In [37]:
%%time
## Transfer_pre 2 ##
cudf_data['Avg. Avg U Kbps'] = data['Avg. Avg U Kbps']

CPU times: user 920 μs, sys: 129 μs, total: 1.05 ms
Wall time: 812 μs


In [38]:
%%time
## Transfer_pre 2 ##
cudf_data['Avg. Avg D Kbps'] = data['Avg. Avg D Kbps']

CPU times: user 1.57 ms, sys: 0 ns, total: 1.57 ms
Wall time: 959 μs


In [39]:
%%time
## Transfer_pre 2 ##
cudf_data['Avg Lat Ms'] = data['Avg Lat Ms']

CPU times: user 1.02 ms, sys: 0 ns, total: 1.02 ms
Wall time: 789 μs


In [40]:
%%time
## Transfer_pre 2 ##
cudf_data['Avg. Pop2005'] = data['Avg. Pop2005']

CPU times: user 1.07 ms, sys: 0 ns, total: 1.07 ms
Wall time: 817 μs


In [41]:
%%time
## Transfer_pre 2 ##
cudf_data['Rank Upload'] = data['Rank Upload']

CPU times: user 1.08 ms, sys: 150 μs, total: 1.23 ms
Wall time: 893 μs


In [42]:
%%time
## Transfer_pre 2 ##
cudf_data['Rank Download'] = data['Rank Download']

CPU times: user 1.16 ms, sys: 0 ns, total: 1.16 ms
Wall time: 761 μs


In [43]:
%%time
## Transfer_pre 2 ##
cudf_data['Rank Latency'] = data['Rank Latency']

CPU times: user 735 μs, sys: 0 ns, total: 735 μs
Wall time: 610 μs


In [44]:
%%time
### cell 2 ###

Mobile_df = pd.DataFrame([],columns=data.columns)
Broadband_df = pd.DataFrame([],columns=data.columns)

def col_name_corrections(df,correction_pair):
    if set(df.columns).intersection(set(correction_pair.keys())):
        df.rename(columns=correction_pair,inplace=True)
    return df

for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        meta_info = filename.split('/')[-1]
        data = pd.read_csv(dirname+'/'+filename,thousands=r',').convert_dtypes()
        data = col_name_corrections(data,{'Number of Record':'Number of Records'})
        data['Year'] = np.int64(re.search('year_(.*)_quarter',meta_info).group(1))
        data['Quarter'] = np.int64(re.search('quarter_(.*).csv',meta_info).group(1))
        if 'mobile' in meta_info:
            Mobile_df = pd.concat([Mobile_df,data])
        else:
            Broadband_df = pd.concat([Broadband_df,data]) 
print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df = Mobile_df.astype({'Year':np.int64,'Quarter':np.int64})
Broadband_df = Broadband_df.astype({'Year':np.int64,'Quarter':np.int64})
Mobile_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)
Broadband_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)

(2597, 13)
(2487, 13)
CPU times: user 92.9 ms, sys: 0 ns, total: 92.9 ms
Wall time: 91.5 ms


<timed exec>:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
<timed exec>:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


In [45]:
cudf_Broadband_df = cudf.DataFrame()

In [46]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 1.61 ms, sys: 221 μs, total: 1.83 ms
Wall time: 1.55 ms


In [47]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Number of Records'] = Broadband_df['Number of Records']

CPU times: user 7.2 ms, sys: 3.91 ms, total: 11.1 ms
Wall time: 9.91 ms


In [48]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Devices'] = Broadband_df['Devices']

CPU times: user 3.28 ms, sys: 452 μs, total: 3.73 ms
Wall time: 2.54 ms


In [49]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Tests'] = Broadband_df['Tests']

CPU times: user 3.03 ms, sys: 77 μs, total: 3.11 ms
Wall time: 2.26 ms


In [50]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Avg. Avg U Kbps'] = Broadband_df['Avg. Avg U Kbps']

CPU times: user 3.63 ms, sys: 0 ns, total: 3.63 ms
Wall time: 2.42 ms


In [51]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Avg. Avg D Kbps'] = Broadband_df['Avg. Avg D Kbps']

CPU times: user 2.61 ms, sys: 359 μs, total: 2.97 ms
Wall time: 2.11 ms


In [52]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Avg Lat Ms'] = Broadband_df['Avg Lat Ms']

CPU times: user 3.81 ms, sys: 0 ns, total: 3.81 ms
Wall time: 2.59 ms


In [53]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Avg. Pop2005'] = Broadband_df['Avg. Pop2005']

CPU times: user 2.99 ms, sys: 0 ns, total: 2.99 ms
Wall time: 2.11 ms


In [54]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Rank Upload'] = Broadband_df['Rank Upload']

CPU times: user 2.39 ms, sys: 326 μs, total: 2.71 ms
Wall time: 1.99 ms


In [55]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Rank Download'] = Broadband_df['Rank Download']

CPU times: user 2.02 ms, sys: 0 ns, total: 2.02 ms
Wall time: 1.72 ms


In [56]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Rank Latency'] = Broadband_df['Rank Latency']

CPU times: user 3.34 ms, sys: 0 ns, total: 3.34 ms
Wall time: 2.3 ms


In [57]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Year'] = Broadband_df['Year']

CPU times: user 1.95 ms, sys: 245 μs, total: 2.19 ms
Wall time: 1.86 ms


In [58]:
%%time
## Transfer_post 2 ##
cudf_Broadband_df['Quarter'] = Broadband_df['Quarter']

CPU times: user 0 ns, sys: 3.22 ms, total: 3.22 ms
Wall time: 2.34 ms


In [59]:
cudf_Mobile_df = cudf.DataFrame()

In [60]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 2.59 ms, sys: 0 ns, total: 2.59 ms
Wall time: 1.91 ms


In [61]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 2.8 ms, sys: 345 μs, total: 3.15 ms
Wall time: 2.24 ms


In [62]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 0 ns, sys: 3.51 ms, total: 3.51 ms
Wall time: 2.35 ms


In [63]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 2.46 ms, sys: 340 μs, total: 2.8 ms
Wall time: 2.09 ms


In [64]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 2.3 ms, sys: 320 μs, total: 2.62 ms
Wall time: 2 ms


In [65]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 3.43 ms, sys: 0 ns, total: 3.43 ms
Wall time: 2.7 ms


In [66]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 3.09 ms, sys: 431 μs, total: 3.52 ms
Wall time: 2.42 ms


In [67]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 2.64 ms, sys: 0 ns, total: 2.64 ms
Wall time: 1.96 ms


In [68]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 3.25 ms, sys: 0 ns, total: 3.25 ms
Wall time: 2.31 ms


In [69]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 2.29 ms, sys: 319 μs, total: 2.61 ms
Wall time: 2.11 ms


In [70]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 3.56 ms, sys: 0 ns, total: 3.56 ms
Wall time: 2.54 ms


In [71]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 3.29 ms, sys: 0 ns, total: 3.29 ms
Wall time: 2.37 ms


In [72]:
%%time
## Transfer_post 2 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 1.99 ms, sys: 277 μs, total: 2.27 ms
Wall time: 1.91 ms


It can be seen that there are more broadband rows than Mobile rows. This is a point that should be noted each row corresponds to a country's statistics in the particular year and quarter. Missing data indicates lack of speed test data from the country.

In [73]:
cudf_Mobile_df = cudf.DataFrame()

In [74]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 1.69 ms, sys: 0 ns, total: 1.69 ms
Wall time: 1.39 ms


In [75]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 2.48 ms, sys: 173 μs, total: 2.65 ms
Wall time: 1.91 ms


In [76]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 1.95 ms, sys: 0 ns, total: 1.95 ms
Wall time: 1.59 ms


In [77]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 2.71 ms, sys: 0 ns, total: 2.71 ms
Wall time: 2.02 ms


In [78]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 2.3 ms, sys: 321 μs, total: 2.62 ms
Wall time: 1.98 ms


In [79]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 0 ns, sys: 3.89 ms, total: 3.89 ms
Wall time: 2.44 ms


In [80]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 1.84 ms, sys: 0 ns, total: 1.84 ms
Wall time: 1.66 ms


In [81]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 2.39 ms, sys: 339 μs, total: 2.73 ms
Wall time: 1.96 ms


In [82]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 1.2 ms, sys: 2.56 ms, total: 3.76 ms
Wall time: 2.53 ms


In [83]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 2.48 ms, sys: 0 ns, total: 2.48 ms
Wall time: 1.89 ms


In [84]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 2.41 ms, sys: 320 μs, total: 2.73 ms
Wall time: 1.9 ms


In [85]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 1.97 ms, sys: 0 ns, total: 1.97 ms
Wall time: 1.68 ms


In [86]:
%%time
## Transfer_pre 3 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 2.95 ms, sys: 0 ns, total: 2.95 ms
Wall time: 2.2 ms


In [87]:
%%time
### cell 3 ###

Mobile_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2487 entries, 0 to 232
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               2487 non-null   string
 1   Number of Records  2487 non-null   Int64 
 2   Devices            2487 non-null   Int64 
 3   Tests              2487 non-null   Int64 
 4   Avg. Avg U Kbps    2487 non-null   Int64 
 5   Avg. Avg D Kbps    2487 non-null   Int64 
 6   Avg Lat Ms         2487 non-null   Int64 
 7   Avg. Pop2005       2487 non-null   Int64 
 8   Rank Upload        2487 non-null   Int64 
 9   Rank Download      2487 non-null   Int64 
 10  Rank Latency       2487 non-null   Int64 
 11  Year               2487 non-null   int64 
 12  Quarter            2487 non-null   int64 
dtypes: Int64(10), int64(2), string(1)
memory usage: 296.3 KB
CPU times: user 6.08 ms, sys: 98 μs, total: 6.18 ms
Wall time: 4.82 ms


# -- STEFANOS -- Replicate Data

In [88]:
cudf_Mobile_df = cudf.DataFrame()

In [89]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 1.59 ms, sys: 0 ns, total: 1.59 ms
Wall time: 1.36 ms


In [90]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 2.48 ms, sys: 354 μs, total: 2.83 ms
Wall time: 2.09 ms


In [91]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 3.52 ms, sys: 158 μs, total: 3.68 ms
Wall time: 2.53 ms


In [92]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 4.09 ms, sys: 5 μs, total: 4.1 ms
Wall time: 2.72 ms


In [93]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 0 ns, sys: 3.36 ms, total: 3.36 ms
Wall time: 2.34 ms


In [94]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 3.65 ms, sys: 526 μs, total: 4.18 ms
Wall time: 2.79 ms


In [95]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 0 ns, sys: 3.92 ms, total: 3.92 ms
Wall time: 2.6 ms


In [96]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 4.68 ms, sys: 91 μs, total: 4.77 ms
Wall time: 3.45 ms


In [97]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 3.48 ms, sys: 0 ns, total: 3.48 ms
Wall time: 2.44 ms


In [98]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 3.22 ms, sys: 471 μs, total: 3.69 ms
Wall time: 2.65 ms


In [99]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 3.44 ms, sys: 144 μs, total: 3.59 ms
Wall time: 2.53 ms


In [100]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 0 ns, sys: 3.44 ms, total: 3.44 ms
Wall time: 2.5 ms


In [101]:
%%time
## Transfer_pre 4 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 3.3 ms, sys: 0 ns, total: 3.3 ms
Wall time: 2.41 ms


In [102]:
%%time
### cell 4 ###

# factor = 2000
factor = 2000
Mobile_df = pd.concat([Mobile_df]*factor, ignore_index=True)
Mobile_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4974000 entries, 0 to 4973999
Data columns (total 13 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   Name               string
 1   Number of Records  Int64 
 2   Devices            Int64 
 3   Tests              Int64 
 4   Avg. Avg U Kbps    Int64 
 5   Avg. Avg D Kbps    Int64 
 6   Avg Lat Ms         Int64 
 7   Avg. Pop2005       Int64 
 8   Rank Upload        Int64 
 9   Rank Download      Int64 
 10  Rank Latency       Int64 
 11  Year               int64 
 12  Quarter            int64 
dtypes: Int64(10), int64(2), string(1)
memory usage: 540.8 MB
CPU times: user 371 ms, sys: 190 ms, total: 560 ms
Wall time: 557 ms


In [103]:
cudf_Mobile_df = cudf.DataFrame()

In [104]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 466 ms, sys: 46.5 ms, total: 512 ms
Wall time: 508 ms


In [105]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 13.7 ms, sys: 4.32 ms, total: 18.1 ms
Wall time: 15.8 ms


In [106]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 17.1 ms, sys: 146 μs, total: 17.3 ms
Wall time: 15.4 ms


In [107]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 13.4 ms, sys: 3.6 ms, total: 17 ms
Wall time: 15.5 ms


In [108]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 16 ms, sys: 619 μs, total: 16.6 ms
Wall time: 15.3 ms


In [109]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 13.6 ms, sys: 3.62 ms, total: 17.2 ms
Wall time: 15.7 ms


In [110]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 17.2 ms, sys: 157 μs, total: 17.4 ms
Wall time: 15.4 ms


In [111]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 16.5 ms, sys: 0 ns, total: 16.5 ms
Wall time: 15.1 ms


In [112]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 16.9 ms, sys: 100 μs, total: 17 ms
Wall time: 15.1 ms


In [113]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 16.8 ms, sys: 97 μs, total: 16.9 ms
Wall time: 15.2 ms


In [114]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 16.4 ms, sys: 33 μs, total: 16.5 ms
Wall time: 15 ms


In [115]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 10.6 ms, sys: 32.9 ms, total: 43.5 ms
Wall time: 42.1 ms


In [116]:
%%time
## Transfer_post 4 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 11.1 ms, sys: 0 ns, total: 11.1 ms
Wall time: 9.28 ms


In [117]:
cudf_Broadband_df = cudf.DataFrame()

In [118]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 1.8 ms, sys: 116 μs, total: 1.91 ms
Wall time: 1.65 ms


In [119]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Number of Records'] = Broadband_df['Number of Records']

CPU times: user 2.8 ms, sys: 0 ns, total: 2.8 ms
Wall time: 2.08 ms


In [120]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Devices'] = Broadband_df['Devices']

CPU times: user 0 ns, sys: 3.94 ms, total: 3.94 ms
Wall time: 2.64 ms


In [121]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Tests'] = Broadband_df['Tests']

CPU times: user 3.36 ms, sys: 559 μs, total: 3.92 ms
Wall time: 2.66 ms


In [122]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Avg. Avg U Kbps'] = Broadband_df['Avg. Avg U Kbps']

CPU times: user 2.86 ms, sys: 0 ns, total: 2.86 ms
Wall time: 2.04 ms


In [123]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Avg. Avg D Kbps'] = Broadband_df['Avg. Avg D Kbps']

CPU times: user 2.88 ms, sys: 0 ns, total: 2.88 ms
Wall time: 1.97 ms


In [124]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Avg Lat Ms'] = Broadband_df['Avg Lat Ms']

CPU times: user 3.46 ms, sys: 0 ns, total: 3.46 ms
Wall time: 2.29 ms


In [125]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Avg. Pop2005'] = Broadband_df['Avg. Pop2005']

CPU times: user 2.97 ms, sys: 0 ns, total: 2.97 ms
Wall time: 2.2 ms


In [126]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Rank Upload'] = Broadband_df['Rank Upload']

CPU times: user 0 ns, sys: 3.43 ms, total: 3.43 ms
Wall time: 2.4 ms


In [127]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Rank Download'] = Broadband_df['Rank Download']

CPU times: user 2.4 ms, sys: 398 μs, total: 2.8 ms
Wall time: 1.96 ms


In [128]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Rank Latency'] = Broadband_df['Rank Latency']

CPU times: user 2.4 ms, sys: 0 ns, total: 2.4 ms
Wall time: 1.9 ms


In [129]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Year'] = Broadband_df['Year']

CPU times: user 3.63 ms, sys: 0 ns, total: 3.63 ms
Wall time: 2.48 ms


In [130]:
%%time
## Transfer_pre 5 ##
cudf_Broadband_df['Quarter'] = Broadband_df['Quarter']

CPU times: user 3.42 ms, sys: 24 μs, total: 3.45 ms
Wall time: 2.38 ms


In [131]:
%%time
### cell 5 ###

factor = 1000
Broadband_df = pd.concat([Broadband_df]*factor, ignore_index=True)
Broadband_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2597000 entries, 0 to 2596999
Data columns (total 13 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   Name               string
 1   Number of Records  Int64 
 2   Devices            Int64 
 3   Tests              Int64 
 4   Avg. Avg U Kbps    Int64 
 5   Avg. Avg D Kbps    Int64 
 6   Avg Lat Ms         Int64 
 7   Avg. Pop2005       Int64 
 8   Rank Upload        Int64 
 9   Rank Download      Int64 
 10  Rank Latency       Int64 
 11  Year               int64 
 12  Quarter            int64 
dtypes: Int64(10), int64(2), string(1)
memory usage: 282.3 MB
CPU times: user 187 ms, sys: 113 ms, total: 299 ms
Wall time: 297 ms


In [132]:
cudf_Broadband_df = cudf.DataFrame()

In [133]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 244 ms, sys: 19.7 ms, total: 264 ms
Wall time: 262 ms


In [134]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Number of Records'] = Broadband_df['Number of Records']

CPU times: user 11.2 ms, sys: 51 μs, total: 11.3 ms
Wall time: 9.07 ms


In [135]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Devices'] = Broadband_df['Devices']

CPU times: user 10.5 ms, sys: 420 μs, total: 11 ms
Wall time: 9.01 ms


In [136]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Tests'] = Broadband_df['Tests']

CPU times: user 3.46 ms, sys: 7.32 ms, total: 10.8 ms
Wall time: 8.82 ms


In [137]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Avg. Avg U Kbps'] = Broadband_df['Avg. Avg U Kbps']

CPU times: user 7.19 ms, sys: 3.19 ms, total: 10.4 ms
Wall time: 8.74 ms


In [138]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Avg. Avg D Kbps'] = Broadband_df['Avg. Avg D Kbps']

CPU times: user 10.2 ms, sys: 366 μs, total: 10.6 ms
Wall time: 8.94 ms


In [139]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Avg Lat Ms'] = Broadband_df['Avg Lat Ms']

CPU times: user 6.7 ms, sys: 3.82 ms, total: 10.5 ms
Wall time: 8.75 ms


In [140]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Avg. Pop2005'] = Broadband_df['Avg. Pop2005']

CPU times: user 10.1 ms, sys: 337 μs, total: 10.4 ms
Wall time: 8.74 ms


In [141]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Rank Upload'] = Broadband_df['Rank Upload']

CPU times: user 10.5 ms, sys: 412 μs, total: 10.9 ms
Wall time: 8.84 ms


In [142]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Rank Download'] = Broadband_df['Rank Download']

CPU times: user 7.21 ms, sys: 3.2 ms, total: 10.4 ms
Wall time: 8.77 ms


In [143]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Rank Latency'] = Broadband_df['Rank Latency']

CPU times: user 10.8 ms, sys: 467 μs, total: 11.3 ms
Wall time: 9 ms


In [144]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Year'] = Broadband_df['Year']

CPU times: user 4.29 ms, sys: 20.4 ms, total: 24.7 ms
Wall time: 22.9 ms


In [145]:
%%time
## Transfer_post 5 ##
cudf_Broadband_df['Quarter'] = Broadband_df['Quarter']

CPU times: user 3.58 ms, sys: 3.98 ms, total: 7.56 ms
Wall time: 5.86 ms


In [146]:
cudf_Mobile_df = cudf.DataFrame()

In [147]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 457 ms, sys: 47.7 ms, total: 505 ms
Wall time: 502 ms


In [148]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 18.1 ms, sys: 317 μs, total: 18.4 ms
Wall time: 16.3 ms


In [149]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 17.8 ms, sys: 0 ns, total: 17.8 ms
Wall time: 16.3 ms


In [150]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 17.9 ms, sys: 0 ns, total: 17.9 ms
Wall time: 15.9 ms


In [151]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 14.1 ms, sys: 2.98 ms, total: 17.1 ms
Wall time: 15.6 ms


In [152]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 13.9 ms, sys: 3.65 ms, total: 17.5 ms
Wall time: 15.8 ms


In [153]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 13.9 ms, sys: 2.95 ms, total: 16.9 ms
Wall time: 15.5 ms


In [154]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 17.4 ms, sys: 208 μs, total: 17.6 ms
Wall time: 15.7 ms


In [155]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 17.2 ms, sys: 155 μs, total: 17.3 ms
Wall time: 15.6 ms


In [156]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 17.1 ms, sys: 140 μs, total: 17.2 ms
Wall time: 15.6 ms


In [157]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 17.3 ms, sys: 178 μs, total: 17.5 ms
Wall time: 15.8 ms


In [158]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 10.7 ms, sys: 446 μs, total: 11.2 ms
Wall time: 9.23 ms


In [159]:
%%time
## Transfer_pre 6 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 11.6 ms, sys: 0 ns, total: 11.6 ms
Wall time: 9.46 ms


In [160]:
%%time
### cell 6 ###

Mobile_df.head()

CPU times: user 239 μs, sys: 0 ns, total: 239 μs
Wall time: 243 μs


,Name,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
0,Afghanistan,789,1241,3541,3099,6387,70,25067407,220,220,32,2020,1
1,Albania,1626,4513,7228,11414,40238,26,3153731,75,42,197,2020,1
2,Algeria,14174,48699,124160,6150,7724,65,32854159,197,213,35,2020,1
3,American Samoa,23,33,63,8488,22792,62,64051,156,111,38,2020,1
4,Andorra,45,83,110,18160,72764,39,73483,9,6,112,2020,1


In [161]:
cudf_Broadband_df = cudf.DataFrame()

In [162]:
%%time
## Transfer_pre 7 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 252 ms, sys: 16.1 ms, total: 268 ms
Wall time: 265 ms


In [163]:
%%time
### cell 7 ###

unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

CPU times: user 287 ms, sys: 55.2 ms, total: 342 ms
Wall time: 339 ms


,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
Name,,,,,,,,,,,,
Afghanistan,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000
Albania,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000
Algeria,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000
American Samoa,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000
Andorra,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000,11000


In [164]:
cudf_unique_countries_broadband = cudf.DataFrame()

In [165]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Number of Records'] = unique_countries_broadband['Number of Records']

CPU times: user 1.7 ms, sys: 289 μs, total: 1.99 ms
Wall time: 1.59 ms


In [166]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Devices'] = unique_countries_broadband['Devices']

CPU times: user 4.05 ms, sys: 691 μs, total: 4.74 ms
Wall time: 3.53 ms


In [167]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Tests'] = unique_countries_broadband['Tests']

CPU times: user 4.56 ms, sys: 83 μs, total: 4.64 ms
Wall time: 2.83 ms


In [168]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Avg. Avg U Kbps'] = unique_countries_broadband['Avg. Avg U Kbps']

CPU times: user 3.77 ms, sys: 0 ns, total: 3.77 ms
Wall time: 2.67 ms


In [169]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Avg. Avg D Kbps'] = unique_countries_broadband['Avg. Avg D Kbps']

CPU times: user 3.85 ms, sys: 0 ns, total: 3.85 ms
Wall time: 2.75 ms


In [170]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Avg Lat Ms'] = unique_countries_broadband['Avg Lat Ms']

CPU times: user 3.21 ms, sys: 548 μs, total: 3.75 ms
Wall time: 2.59 ms


In [171]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Avg. Pop2005'] = unique_countries_broadband['Avg. Pop2005']

CPU times: user 327 μs, sys: 3.43 ms, total: 3.76 ms
Wall time: 2.57 ms


In [172]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Rank Upload'] = unique_countries_broadband['Rank Upload']

CPU times: user 3.6 ms, sys: 0 ns, total: 3.6 ms
Wall time: 2.43 ms


In [173]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Rank Download'] = unique_countries_broadband['Rank Download']

CPU times: user 1.53 ms, sys: 2.53 ms, total: 4.06 ms
Wall time: 2.76 ms


In [174]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Rank Latency'] = unique_countries_broadband['Rank Latency']

CPU times: user 3.3 ms, sys: 566 μs, total: 3.87 ms
Wall time: 2.62 ms


In [175]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Year'] = unique_countries_broadband['Year']

CPU times: user 2.79 ms, sys: 134 μs, total: 2.93 ms
Wall time: 2.49 ms


In [176]:
%%time
## Transfer_post 7 ##
cudf_unique_countries_broadband['Quarter'] = unique_countries_broadband['Quarter']

CPU times: user 287 μs, sys: 4.13 ms, total: 4.41 ms
Wall time: 2.9 ms


In [177]:
cudf_Mobile_df = cudf.DataFrame()

In [178]:
%%time
## Transfer_pre 8 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 480 ms, sys: 31.8 ms, total: 512 ms
Wall time: 507 ms


In [179]:
%%time
### cell 8 ###

unique_countries_mobile = Mobile_df.groupby('Name').count()
unique_countries_mobile.head()

CPU times: user 579 ms, sys: 71.5 ms, total: 651 ms
Wall time: 646 ms


,Number of Records,Devices,Tests,Avg. Avg U Kbps,Avg. Avg D Kbps,Avg Lat Ms,Avg. Pop2005,Rank Upload,Rank Download,Rank Latency,Year,Quarter
Name,,,,,,,,,,,,
Afghanistan,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000
Albania,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000
Algeria,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000
American Samoa,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000
Andorra,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000,22000


In [180]:
cudf_unique_countries_mobile = cudf.DataFrame()

In [181]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Number of Records'] = unique_countries_mobile['Number of Records']

CPU times: user 1.75 ms, sys: 288 μs, total: 2.03 ms
Wall time: 1.59 ms


In [182]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Devices'] = unique_countries_mobile['Devices']

CPU times: user 0 ns, sys: 4.27 ms, total: 4.27 ms
Wall time: 2.94 ms


In [183]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Tests'] = unique_countries_mobile['Tests']

CPU times: user 4.93 ms, sys: 144 μs, total: 5.08 ms
Wall time: 3.39 ms


In [184]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Avg. Avg U Kbps'] = unique_countries_mobile['Avg. Avg U Kbps']

CPU times: user 3.91 ms, sys: 0 ns, total: 3.91 ms
Wall time: 2.91 ms


In [185]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Avg. Avg D Kbps'] = unique_countries_mobile['Avg. Avg D Kbps']

CPU times: user 2.89 ms, sys: 479 μs, total: 3.36 ms
Wall time: 2.57 ms


In [186]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Avg Lat Ms'] = unique_countries_mobile['Avg Lat Ms']

CPU times: user 4.09 ms, sys: 4 μs, total: 4.1 ms
Wall time: 2.79 ms


In [187]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Avg. Pop2005'] = unique_countries_mobile['Avg. Pop2005']

CPU times: user 4.35 ms, sys: 46 μs, total: 4.39 ms
Wall time: 2.96 ms


In [188]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Rank Upload'] = unique_countries_mobile['Rank Upload']

CPU times: user 3.84 ms, sys: 0 ns, total: 3.84 ms
Wall time: 2.67 ms


In [189]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Rank Download'] = unique_countries_mobile['Rank Download']

CPU times: user 2.98 ms, sys: 494 μs, total: 3.48 ms
Wall time: 2.45 ms


In [190]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Rank Latency'] = unique_countries_mobile['Rank Latency']

CPU times: user 4.1 ms, sys: 185 μs, total: 4.28 ms
Wall time: 2.85 ms


In [191]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Year'] = unique_countries_mobile['Year']

CPU times: user 4.54 ms, sys: 76 μs, total: 4.61 ms
Wall time: 3.04 ms


In [192]:
%%time
## Transfer_post 8 ##
cudf_unique_countries_mobile['Quarter'] = unique_countries_mobile['Quarter']

CPU times: user 659 μs, sys: 3.51 ms, total: 4.17 ms
Wall time: 3.13 ms


## Insights

The following countries do have mobile speedtest data for all the years and quarters, thereby making less than 10 reports (one per quarter).

In [193]:
cudf_Mobile_df = cudf.DataFrame()

In [194]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 477 ms, sys: 28 ms, total: 505 ms
Wall time: 501 ms


In [195]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 18.2 ms, sys: 0 ns, total: 18.2 ms
Wall time: 15.8 ms


In [196]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 16.8 ms, sys: 0 ns, total: 16.8 ms
Wall time: 15.4 ms


In [197]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 13.8 ms, sys: 3.01 ms, total: 16.8 ms
Wall time: 15.3 ms


In [198]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 16.8 ms, sys: 86 μs, total: 16.9 ms
Wall time: 15.1 ms


In [199]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 16.5 ms, sys: 33 μs, total: 16.5 ms
Wall time: 15.1 ms


In [200]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 17.1 ms, sys: 122 μs, total: 17.2 ms
Wall time: 15.5 ms


In [201]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 16.1 ms, sys: 628 μs, total: 16.8 ms
Wall time: 15.1 ms


In [202]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 17 ms, sys: 0 ns, total: 17 ms
Wall time: 15.3 ms


In [203]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 17.2 ms, sys: 138 μs, total: 17.3 ms
Wall time: 15.2 ms


In [204]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 16.8 ms, sys: 74 μs, total: 16.8 ms
Wall time: 15.1 ms


In [205]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 10.5 ms, sys: 373 μs, total: 10.9 ms
Wall time: 9.17 ms


In [206]:
%%time
## Transfer_pre 9 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 10.1 ms, sys: 312 μs, total: 10.4 ms
Wall time: 9.03 ms


In [207]:
%%time
### cell 9 ###

# Check for missing values
Mobile_df.isna().any()

CPU times: user 185 ms, sys: 11.8 ms, total: 197 ms
Wall time: 193 ms


Name                 False
Number of Records    False
Devices              False
Tests                False
Avg. Avg U Kbps      False
Avg. Avg D Kbps      False
Avg Lat Ms           False
Avg. Pop2005         False
Rank Upload          False
Rank Download        False
Rank Latency         False
Year                 False
Quarter              False
dtype: bool

In [208]:
cudf_Broadband_df = cudf.DataFrame()

In [209]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 243 ms, sys: 11.9 ms, total: 255 ms
Wall time: 254 ms


In [210]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Number of Records'] = Broadband_df['Number of Records']

CPU times: user 3.44 ms, sys: 7.63 ms, total: 11.1 ms
Wall time: 9.06 ms


In [211]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Devices'] = Broadband_df['Devices']

CPU times: user 10.3 ms, sys: 338 μs, total: 10.7 ms
Wall time: 8.66 ms


In [212]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Tests'] = Broadband_df['Tests']

CPU times: user 10.2 ms, sys: 321 μs, total: 10.5 ms
Wall time: 8.6 ms


In [213]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Avg. Avg U Kbps'] = Broadband_df['Avg. Avg U Kbps']

CPU times: user 9.73 ms, sys: 244 μs, total: 9.97 ms
Wall time: 8.7 ms


In [214]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Avg. Avg D Kbps'] = Broadband_df['Avg. Avg D Kbps']

CPU times: user 6.1 ms, sys: 3.76 ms, total: 9.86 ms
Wall time: 8.47 ms


In [215]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Avg Lat Ms'] = Broadband_df['Avg Lat Ms']

CPU times: user 9.57 ms, sys: 220 μs, total: 9.79 ms
Wall time: 8.32 ms


In [216]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Avg. Pop2005'] = Broadband_df['Avg. Pop2005']

CPU times: user 9.74 ms, sys: 246 μs, total: 9.99 ms
Wall time: 8.31 ms


In [217]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Rank Upload'] = Broadband_df['Rank Upload']

CPU times: user 6.44 ms, sys: 3.81 ms, total: 10.3 ms
Wall time: 8.89 ms


In [218]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Rank Download'] = Broadband_df['Rank Download']

CPU times: user 10.1 ms, sys: 307 μs, total: 10.4 ms
Wall time: 9.06 ms


In [219]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Rank Latency'] = Broadband_df['Rank Latency']

CPU times: user 9.81 ms, sys: 257 μs, total: 10.1 ms
Wall time: 8.61 ms


In [220]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Year'] = Broadband_df['Year']

CPU times: user 6.94 ms, sys: 0 ns, total: 6.94 ms
Wall time: 5.46 ms


In [221]:
%%time
## Transfer_pre 10 ##
cudf_Broadband_df['Quarter'] = Broadband_df['Quarter']

CPU times: user 6.77 ms, sys: 331 μs, total: 7.1 ms
Wall time: 5.55 ms


In [222]:
%%time
### cell 10 ###

Broadband_df.isna().any()

CPU times: user 101 ms, sys: 7.41 ms, total: 109 ms
Wall time: 106 ms


Name                 False
Number of Records    False
Devices              False
Tests                False
Avg. Avg U Kbps      False
Avg. Avg D Kbps      False
Avg Lat Ms           False
Avg. Pop2005         False
Rank Upload          False
Rank Download        False
Rank Latency         False
Year                 False
Quarter              False
dtype: bool

In [223]:
cudf_Mobile_df = cudf.DataFrame()

In [224]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 475 ms, sys: 32.3 ms, total: 507 ms
Wall time: 500 ms


In [225]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Number of Records'] = Mobile_df['Number of Records']

CPU times: user 17.6 ms, sys: 202 μs, total: 17.8 ms
Wall time: 15.7 ms


In [226]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Devices'] = Mobile_df['Devices']

CPU times: user 17.2 ms, sys: 128 μs, total: 17.3 ms
Wall time: 15.4 ms


In [227]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Tests'] = Mobile_df['Tests']

CPU times: user 17.1 ms, sys: 123 μs, total: 17.3 ms
Wall time: 15.1 ms


In [228]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 16.9 ms, sys: 92 μs, total: 17 ms
Wall time: 15.3 ms


In [229]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 13.5 ms, sys: 3.65 ms, total: 17.1 ms
Wall time: 15.1 ms


In [230]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 16.9 ms, sys: 92 μs, total: 17 ms
Wall time: 15.1 ms


In [231]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Avg. Pop2005'] = Mobile_df['Avg. Pop2005']

CPU times: user 17 ms, sys: 107 μs, total: 17.1 ms
Wall time: 15.2 ms


In [232]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Rank Upload'] = Mobile_df['Rank Upload']

CPU times: user 15.9 ms, sys: 552 μs, total: 16.5 ms
Wall time: 15 ms


In [233]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Rank Download'] = Mobile_df['Rank Download']

CPU times: user 16.5 ms, sys: 32 μs, total: 16.5 ms
Wall time: 15.2 ms


In [234]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Rank Latency'] = Mobile_df['Rank Latency']

CPU times: user 17.3 ms, sys: 0 ns, total: 17.3 ms
Wall time: 15.1 ms


In [235]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Year'] = Mobile_df['Year']

CPU times: user 10.2 ms, sys: 302 μs, total: 10.5 ms
Wall time: 9.2 ms


In [236]:
%%time
## Transfer_pre 11 ##
cudf_Mobile_df['Quarter'] = Mobile_df['Quarter']

CPU times: user 10.8 ms, sys: 0 ns, total: 10.8 ms
Wall time: 9.27 ms


In [237]:
%%time
### cell 11 ###

Mobile_df.shape

CPU times: user 13 μs, sys: 0 ns, total: 13 μs
Wall time: 16 μs


(4974000, 13)

In [238]:
cudf_unique_countries_mobile = cudf.DataFrame()

In [239]:
%%time
## Transfer_pre 12 ##
cudf_unique_countries_mobile['Year'] = unique_countries_mobile['Year']

CPU times: user 2.01 ms, sys: 0 ns, total: 2.01 ms
Wall time: 1.37 ms


In [240]:
%%time
### cell 12 ###

unique_countries_mobile[unique_countries_mobile.Year < 10]['Year']

CPU times: user 1.72 ms, sys: 0 ns, total: 1.72 ms
Wall time: 1.27 ms


Series([], Name: Year, dtype: int64)

In [241]:
cudf_unique_countries_broadband = cudf.DataFrame()

In [242]:
%%time
## Transfer_pre 13 ##
cudf_unique_countries_broadband['Year'] = unique_countries_broadband['Year']

CPU times: user 1.76 ms, sys: 0 ns, total: 1.76 ms
Wall time: 1.23 ms


In [243]:
%%time
### cell 13 ###

unique_countries_broadband[unique_countries_broadband.Year < 10]['Year']

CPU times: user 1.51 ms, sys: 0 ns, total: 1.51 ms
Wall time: 1.06 ms


Series([], Name: Year, dtype: int64)

## Raw Download Speed Visualization

This visualization can be used to show change of values per country. The improvement values cannot be understood by laymen because an improvement of 50 Kbps **national average** (given) means differnt things to different countries based on economy, population, GDP, Infrastructure, etc.

In [244]:
cudf_Broadband_df = cudf.DataFrame()

In [245]:
%%time
## Transfer_pre 14 ##
cudf_Broadband_df['Avg. Avg D Kbps'] = Broadband_df['Avg. Avg D Kbps']

CPU times: user 9.76 ms, sys: 244 μs, total: 10 ms
Wall time: 8.37 ms


In [246]:
%%time
## Transfer_pre 14 ##
cudf_Broadband_df['Avg Lat Ms'] = Broadband_df['Avg Lat Ms']

CPU times: user 10.1 ms, sys: 0 ns, total: 10.1 ms
Wall time: 8.46 ms


In [247]:
%%time
## Transfer_pre 14 ##
cudf_Broadband_df['Avg. Avg U Kbps'] = Broadband_df['Avg. Avg U Kbps']

CPU times: user 7.23 ms, sys: 3.32 ms, total: 10.5 ms
Wall time: 8.44 ms


In [248]:
%%time
## Transfer_pre 14 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 244 ms, sys: 15 ms, total: 259 ms
Wall time: 255 ms


In [249]:
cudf_Mobile_df = cudf.DataFrame()

In [250]:
%%time
## Transfer_pre 14 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 16.3 ms, sys: 0 ns, total: 16.3 ms
Wall time: 15.6 ms


In [251]:
%%time
## Transfer_pre 14 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 17.5 ms, sys: 178 μs, total: 17.7 ms
Wall time: 15.7 ms


In [252]:
%%time
## Transfer_pre 14 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 16.5 ms, sys: 640 μs, total: 17.1 ms
Wall time: 15.6 ms


In [253]:
%%time
## Transfer_pre 14 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 470 ms, sys: 39.2 ms, total: 509 ms
Wall time: 505 ms


In [254]:
%%time
### cell 14 ###

Mobile_Stats = Mobile_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: list(x)[-1] - list(x)[0])
)
Broadband_Stats = Broadband_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: list(x)[-1] - list(x)[0])
)
# fig = px.histogram(Mobile_Stats['Change_Download'],title='Frequency distribution of Mobile Speed change',
#                    labels={'count':'Frequency','value':'$\Delta Speed (Kpbs)$','variable':'property'},
#                    nbins=100)
# fig.show()

# fig = px.histogram(Broadband_Stats['Change_Download'],title='Frequency distribution of Broadband Speed change',
#                    labels={'count':'Frequency','value':'$\Delta Speed (Kpbs)$','variable':'property'},
#                    nbins=100)
# fig.show()
Total_Stats = pd.concat([Broadband_Stats['Change_Download'],Mobile_Stats['Change_Download']],axis=1)
Total_Stats.columns=['Mobile','Broadband']

# STEFANOS: Disable plotting
# fig = go.Figure(data=[go.Histogram(x=Broadband_Stats['Change_Download'],opacity=0.65,name='Broadband')])
# fig.add_trace(go.Histogram(x=Mobile_Stats['Change_Download'],opacity=0.65,name='Mobile'))
# fig.update_layout(barmode='overlay',
#                   title='Frequency Distribution of Speed change',
#                   xaxis_title="$\Delta\ Speed\ (Kbps)$", yaxis_title="Number of Countries",
#                   legend_title='Color')
# fig.show()

CPU times: user 4.42 s, sys: 264 ms, total: 4.68 s
Wall time: 4.64 s


In [255]:
cudf_Broadband_Stats = cudf.DataFrame()

In [256]:
%%time
## Transfer_post 14 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 1.42 ms, sys: 178 μs, total: 1.59 ms
Wall time: 1.3 ms


In [257]:
%%time
## Transfer_post 14 ##
cudf_Broadband_Stats['Change_Upload'] = Broadband_Stats['Change_Upload']

CPU times: user 2.84 ms, sys: 0 ns, total: 2.84 ms
Wall time: 2.37 ms


In [258]:
%%time
## Transfer_post 14 ##
cudf_Broadband_Stats['Change_Latency'] = Broadband_Stats['Change_Latency']

CPU times: user 3.84 ms, sys: 0 ns, total: 3.84 ms
Wall time: 2.71 ms


In [259]:
cudf_Mobile_Stats = cudf.DataFrame()

In [260]:
%%time
## Transfer_post 14 ##
cudf_Mobile_Stats['Change_Download'] = Mobile_Stats['Change_Download']

CPU times: user 0 ns, sys: 1.77 ms, total: 1.77 ms
Wall time: 1.22 ms


In [261]:
%%time
## Transfer_post 14 ##
cudf_Mobile_Stats['Change_Upload'] = Mobile_Stats['Change_Upload']

CPU times: user 3.26 ms, sys: 412 μs, total: 3.67 ms
Wall time: 2.58 ms


In [262]:
%%time
## Transfer_post 14 ##
cudf_Mobile_Stats['Change_Latency'] = Mobile_Stats['Change_Latency']

CPU times: user 3.5 ms, sys: 156 μs, total: 3.65 ms
Wall time: 2.76 ms


It can see that most of the countries changed between -5000 Kbps to 5000 kbps. A common graph for all the countries is possible but makes it difficult to understand. Therefore it is better we split the countries into different visualizations, for seperate degrees of change

In [263]:
# STEFANOS: Disable plotting
# px.bar(Mobile_Stats,y='Change_Download',labels={'Name':'Country','Change_Download':'Observed Change'},title='Summary of all changes 2020 Q1 - 2022 Q2 ')

In [264]:
cudf_Broadband_Stats = cudf.DataFrame()

In [265]:
%%time
## Transfer_pre 15 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 2.18 ms, sys: 0 ns, total: 2.18 ms
Wall time: 1.49 ms


In [266]:
%%time
### cell 15 ###

#ImprovedCountries_M = Mobile_Stats[(Mobile_Stats['Change_Download'] < 3000) &
#                                (Mobile_Stats['Change_Download'] >0)]
#px.bar(ImprovedCountries_M,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 3000) &
                                (Broadband_Stats['Change_Download'] > 0)]

# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

CPU times: user 1.68 ms, sys: 214 μs, total: 1.9 ms
Wall time: 1.78 ms


In [267]:
cudf_Broadband_Stats = cudf.DataFrame()

In [268]:
%%time
## Transfer_pre 16 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 2.11 ms, sys: 0 ns, total: 2.11 ms
Wall time: 1.2 ms


In [269]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [270]:
%%time
## Transfer_pre 16 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 0 ns, sys: 2.59 ms, total: 2.59 ms
Wall time: 1.8 ms


In [271]:
%%time
## Transfer_pre 16 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 4 ms, sys: 0 ns, total: 4 ms
Wall time: 2.82 ms


In [272]:
%%time
## Transfer_pre 16 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 4.04 ms, sys: 0 ns, total: 4.04 ms
Wall time: 2.82 ms


In [273]:
%%time
### cell 16 ###

#ImprovedCountries2 = Mobile_Stats[(Mobile_Stats['Change_Download'] >= 10000)]
#px.bar(ImprovedCountries2,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')
#ImprovedCountries_M = Mobile_Stats[(Mobile_Stats['Change_Download'] < 8000) &
#                                (Mobile_Stats['Change_Download'] >3000)]
#px.bar(ImprovedCountries_M,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 8000) &
                                (Broadband_Stats['Change_Download'] > 3000)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

CPU times: user 862 μs, sys: 110 μs, total: 972 μs
Wall time: 826 μs


In [274]:
cudf_Broadband_Stats = cudf.DataFrame()

In [275]:
%%time
## Transfer_pre 17 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 2.12 ms, sys: 269 μs, total: 2.39 ms
Wall time: 1.63 ms


In [276]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [277]:
%%time
## Transfer_pre 17 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 2.01 ms, sys: 256 μs, total: 2.27 ms
Wall time: 1.56 ms


In [278]:
%%time
## Transfer_pre 17 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 3.93 ms, sys: 0 ns, total: 3.93 ms
Wall time: 2.78 ms


In [279]:
%%time
## Transfer_pre 17 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 3.57 ms, sys: 0 ns, total: 3.57 ms
Wall time: 2.62 ms


In [280]:
%%time
### cell 17 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 16000) &
                                (Broadband_Stats['Change_Download'] > 8000)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

CPU times: user 1.01 ms, sys: 0 ns, total: 1.01 ms
Wall time: 793 μs


In [281]:
cudf_Broadband_Stats = cudf.DataFrame()

In [282]:
%%time
## Transfer_pre 18 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 1.3 ms, sys: 0 ns, total: 1.3 ms
Wall time: 1.05 ms


In [283]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [284]:
%%time
## Transfer_pre 18 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 2.14 ms, sys: 0 ns, total: 2.14 ms
Wall time: 1.39 ms


In [285]:
%%time
## Transfer_pre 18 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 2.86 ms, sys: 252 μs, total: 3.11 ms
Wall time: 2.18 ms


In [286]:
%%time
## Transfer_pre 18 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 915 μs, sys: 3.67 ms, total: 4.59 ms
Wall time: 2.83 ms


In [287]:
%%time
### cell 18 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 60000) &
                                (Broadband_Stats['Change_Download'] > 16000)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

CPU times: user 913 μs, sys: 116 μs, total: 1.03 ms
Wall time: 810 μs


In [288]:
cudf_Broadband_Stats = cudf.DataFrame()

In [289]:
%%time
## Transfer_pre 19 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 1.18 ms, sys: 150 μs, total: 1.33 ms
Wall time: 1.06 ms


In [290]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [291]:
%%time
## Transfer_pre 19 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 1.2 ms, sys: 152 μs, total: 1.35 ms
Wall time: 1.09 ms


In [292]:
%%time
## Transfer_pre 19 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 3.6 ms, sys: 458 μs, total: 4.06 ms
Wall time: 2.7 ms


In [293]:
%%time
## Transfer_pre 19 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 3.17 ms, sys: 65 μs, total: 3.23 ms
Wall time: 2.5 ms


In [294]:
%%time
### cell 19 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] >= 60000)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

CPU times: user 690 μs, sys: 0 ns, total: 690 μs
Wall time: 576 μs


In [295]:
cudf_Broadband_Stats = cudf.DataFrame()

In [296]:
%%time
## Transfer_pre 20 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 1.26 ms, sys: 85 μs, total: 1.34 ms
Wall time: 1.08 ms


In [297]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [298]:
%%time
## Transfer_pre 20 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 1.61 ms, sys: 204 μs, total: 1.81 ms
Wall time: 1.19 ms


In [299]:
%%time
## Transfer_pre 20 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 2.22 ms, sys: 0 ns, total: 2.22 ms
Wall time: 1.95 ms


In [300]:
%%time
## Transfer_pre 20 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 3.97 ms, sys: 0 ns, total: 3.97 ms
Wall time: 2.52 ms


In [301]:
%%time
### cell 20 ###

#DeterioratedSpeeds = Mobile_Stats[(Mobile_Stats['Change_Download'] < 0 )]
#px.bar(DeterioratedSpeeds,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Decreasing Countries\' download speeds')
ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 0)]
Countries = ImprovedCountries_B.index

# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Speed change',
#                   xaxis_title="Country", yaxis_title="Improved Speed (Kbps)",
#                   legend_title='Color')
# fig.show()

CPU times: user 615 μs, sys: 78 μs, total: 693 μs
Wall time: 574 μs


In [302]:
cudf_Mobile_Stats = cudf.DataFrame()

In [303]:
%%time
## Transfer_pre 21 ##
cudf_Mobile_Stats['Change_Download'] = Mobile_Stats['Change_Download']

CPU times: user 1.17 ms, sys: 149 μs, total: 1.32 ms
Wall time: 1.06 ms


In [304]:
%%time
### cell 21 ###

Mobile_Stats.sort_values(by=['Change_Download'])

CPU times: user 836 μs, sys: 106 μs, total: 942 μs
Wall time: 614 μs


,Change_Download,Change_Upload,Change_Latency
Name,,,
Svalbard,-60722,-7622,26
Cook Islands,-53299,-12437,96
Kiribati,-16481,-6388,323
Cuba,-13664,-3164,22
Lesotho,-10968,149,2
...,...,...,...
Bahrain,112317,7053,1
United Arab Emirates,113202,6452,3
Kuwait,120812,9258,-2


In [305]:
cudf_Broadband_Stats = cudf.DataFrame()

In [306]:
%%time
## Transfer_pre 22 ##
cudf_Broadband_Stats['Change_Download'] = Broadband_Stats['Change_Download']

CPU times: user 2.02 ms, sys: 0 ns, total: 2.02 ms
Wall time: 1.31 ms


In [307]:
%%time
### cell 22 ###

Broadband_Stats.sort_values(by=['Change_Download'])

CPU times: user 815 μs, sys: 0 ns, total: 815 μs
Wall time: 614 μs


,Change_Download,Change_Upload,Change_Latency
Name,,,
Marshall Islands,-9495,-2082,-13
Comoros,-7547,-18156,38
United States Minor Outlying Islands,-7465,-18425,461
Guinea-Bissau,-5363,-2684,30
Cape Verde,-4525,-8908,-10
...,...,...,...
France,106223,82203,-10
Uruguay,109183,21217,-18
China,127467,23571,3


Different graphs are used to show different degrees of Average download speed change for each country. These are the capacities by which countries have changed. 

From the above graphs it can be seen that **China** has improved the most average (boardband) download speed and **Antarctica** has lost the most average download internet speed (broadband). Meanwhile **Korea** has the highest improvement in Mobile internet download speed, while **Cook Islands** has lost the most average mobile download speed

These metrics can be misleading as they show average speed change for the whole country. The infrastructure investment/deterioration of the country can only be known after the number is multiplied by population of the country.

## Percentage Download speed Visualization

In this view, we have normalized all improvements or depreciation to the original value, therefore large countries such as china which already have large infrastructure will be given less weightage and small countries that are developing would be given more preference. This view can be used everywhere as the values are normalized to percentages. Therefore larger internet speed nations will be given lower priority to the smaller speed nations. This is NOT a metric of a nation's total bandwidth as that will require denormalization with the nation's population. This metric can be used to compare the rate of national speed improvement between nations.

In [308]:
cudf_Broadband_df = cudf.DataFrame()

In [309]:
%%time
## Transfer_pre 23 ##
cudf_Broadband_df['Avg. Avg D Kbps'] = Broadband_df['Avg. Avg D Kbps']

CPU times: user 11 ms, sys: 0 ns, total: 11 ms
Wall time: 8.87 ms


In [310]:
%%time
## Transfer_pre 23 ##
cudf_Broadband_df['Avg Lat Ms'] = Broadband_df['Avg Lat Ms']

CPU times: user 7.64 ms, sys: 3.35 ms, total: 11 ms
Wall time: 8.87 ms


In [311]:
%%time
## Transfer_pre 23 ##
cudf_Broadband_df['Avg. Avg U Kbps'] = Broadband_df['Avg. Avg U Kbps']

CPU times: user 10.7 ms, sys: 322 μs, total: 11 ms
Wall time: 9.02 ms


In [312]:
%%time
## Transfer_pre 23 ##
cudf_Broadband_df['Name'] = Broadband_df['Name']

CPU times: user 250 ms, sys: 20.1 ms, total: 270 ms
Wall time: 265 ms


In [313]:
cudf_Mobile_df = cudf.DataFrame()

In [314]:
%%time
## Transfer_pre 23 ##
cudf_Mobile_df['Avg. Avg D Kbps'] = Mobile_df['Avg. Avg D Kbps']

CPU times: user 17.2 ms, sys: 112 μs, total: 17.3 ms
Wall time: 15.2 ms


In [315]:
%%time
## Transfer_pre 23 ##
cudf_Mobile_df['Avg Lat Ms'] = Mobile_df['Avg Lat Ms']

CPU times: user 14.4 ms, sys: 2.79 ms, total: 17.2 ms
Wall time: 15.3 ms


In [316]:
%%time
## Transfer_pre 23 ##
cudf_Mobile_df['Avg. Avg U Kbps'] = Mobile_df['Avg. Avg U Kbps']

CPU times: user 16.8 ms, sys: 58 μs, total: 16.8 ms
Wall time: 15.1 ms


In [317]:
%%time
## Transfer_pre 23 ##
cudf_Mobile_df['Name'] = Mobile_df['Name']

CPU times: user 467 ms, sys: 43.9 ms, total: 511 ms
Wall time: 505 ms


In [318]:
%%time
### cell 23 ###

Mobile_Stats_relative = Mobile_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0])
)
Broadband_Stats_relative = Broadband_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0])
)

CPU times: user 6.08 s, sys: 221 ms, total: 6.3 s
Wall time: 6.25 s


In [319]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [320]:
%%time
## Transfer_post 23 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 7.05 ms, sys: 4.38 ms, total: 11.4 ms
Wall time: 10.9 ms


In [321]:
%%time
## Transfer_post 23 ##
cudf_Broadband_Stats_relative['Change_Upload'] = Broadband_Stats_relative['Change_Upload']

CPU times: user 5.07 ms, sys: 0 ns, total: 5.07 ms
Wall time: 3.91 ms


In [322]:
%%time
## Transfer_post 23 ##
cudf_Broadband_Stats_relative['Change_Latency'] = Broadband_Stats_relative['Change_Latency']

CPU times: user 4.68 ms, sys: 49 μs, total: 4.73 ms
Wall time: 3.5 ms


In [323]:
cudf_Mobile_Stats_relative = cudf.DataFrame()

In [324]:
%%time
## Transfer_post 23 ##
cudf_Mobile_Stats_relative['Change_Download'] = Mobile_Stats_relative['Change_Download']

CPU times: user 2.82 ms, sys: 0 ns, total: 2.82 ms
Wall time: 2.06 ms


In [325]:
%%time
## Transfer_post 23 ##
cudf_Mobile_Stats_relative['Change_Upload'] = Mobile_Stats_relative['Change_Upload']

CPU times: user 1.17 ms, sys: 3.76 ms, total: 4.93 ms
Wall time: 3.67 ms


In [326]:
%%time
## Transfer_post 23 ##
cudf_Mobile_Stats_relative['Change_Latency'] = Mobile_Stats_relative['Change_Latency']

CPU times: user 420 μs, sys: 4.11 ms, total: 4.53 ms
Wall time: 3.57 ms


In [327]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [328]:
%%time
## Transfer_pre 24 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 2.9 ms, sys: 305 μs, total: 3.21 ms
Wall time: 1.84 ms


In [329]:
%%time
## Transfer_pre 24 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 0 ns, sys: 3.76 ms, total: 3.76 ms
Wall time: 2.72 ms


In [330]:
%%time
## Transfer_pre 24 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 4.14 ms, sys: 7 μs, total: 4.15 ms
Wall time: 2.82 ms


In [331]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [332]:
%%time
## Transfer_pre 24 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 2.48 ms, sys: 260 μs, total: 2.74 ms
Wall time: 1.94 ms


In [333]:
%%time
### cell 24 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 2)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats_relative.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Relative Speed change',
#                   xaxis_title="Country", yaxis_title="Relative Improvement of Speed",
#                   legend_title='Color')
# fig.show()

CPU times: user 759 μs, sys: 0 ns, total: 759 μs
Wall time: 685 μs


In [334]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [335]:
%%time
## Transfer_pre 25 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 2.33 ms, sys: 246 μs, total: 2.57 ms
Wall time: 1.92 ms


In [336]:
%%time
## Transfer_pre 25 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 0 ns, sys: 5.15 ms, total: 5.15 ms
Wall time: 3.84 ms


In [337]:
%%time
## Transfer_pre 25 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 4.7 ms, sys: 67 μs, total: 4.76 ms
Wall time: 3.68 ms


In [338]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [339]:
%%time
## Transfer_pre 25 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 4.06 ms, sys: 0 ns, total: 4.06 ms
Wall time: 2.67 ms


In [340]:
%%time
### cell 25 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 1) & (Broadband_Stats_relative['Change_Download'] < 2)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats_relative.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Relative Speed change',
#                   xaxis_title="Country", yaxis_title="Relative Improvement of Speed",
#                   legend_title='Color')
# fig.show()

CPU times: user 751 μs, sys: 80 μs, total: 831 μs
Wall time: 754 μs


In [341]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [342]:
%%time
## Transfer_pre 26 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 4.35 ms, sys: 31 μs, total: 4.38 ms
Wall time: 2.77 ms


In [343]:
%%time
## Transfer_pre 26 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 5.04 ms, sys: 104 μs, total: 5.15 ms
Wall time: 3.76 ms


In [344]:
%%time
## Transfer_pre 26 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 4.54 ms, sys: 50 μs, total: 4.59 ms
Wall time: 3.45 ms


In [345]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [346]:
%%time
## Transfer_pre 26 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 4.03 ms, sys: 0 ns, total: 4.03 ms
Wall time: 2.55 ms


In [347]:
%%time
### cell 26 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.5) & (Broadband_Stats_relative['Change_Download'] < 1)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats_relative.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Relative Speed change',
#                   xaxis_title="Country", yaxis_title="Relative Improvement of Speed",
#                   legend_title='Color')
# fig.show()

CPU times: user 841 μs, sys: 0 ns, total: 841 μs
Wall time: 749 μs


In [348]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [349]:
%%time
## Transfer_pre 27 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 2.3 ms, sys: 244 μs, total: 2.54 ms
Wall time: 1.92 ms


In [350]:
%%time
## Transfer_pre 27 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 5.06 ms, sys: 104 μs, total: 5.16 ms
Wall time: 3.84 ms


In [351]:
%%time
## Transfer_pre 27 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 4.86 ms, sys: 0 ns, total: 4.86 ms
Wall time: 3.75 ms


In [352]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [353]:
%%time
## Transfer_pre 27 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 3.02 ms, sys: 0 ns, total: 3.02 ms
Wall time: 2.16 ms


In [354]:
%%time
### cell 27 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.2) & (Broadband_Stats_relative['Change_Download'] < 0.5)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats_relative.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Relative Speed change',
#                   xaxis_title="Country", yaxis_title="Relative Improvement of Speed",
#                   legend_title='Color')
# fig.show()

CPU times: user 899 μs, sys: 95 μs, total: 994 μs
Wall time: 732 μs


In [355]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [356]:
%%time
## Transfer_pre 28 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 0 ns, sys: 3.74 ms, total: 3.74 ms
Wall time: 2.52 ms


In [357]:
%%time
## Transfer_pre 28 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 3.53 ms, sys: 0 ns, total: 3.53 ms
Wall time: 2.9 ms


In [358]:
%%time
## Transfer_pre 28 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 4.47 ms, sys: 42 μs, total: 4.51 ms
Wall time: 3.46 ms


In [359]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [360]:
%%time
## Transfer_pre 28 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 313 μs, sys: 3.67 ms, total: 3.98 ms
Wall time: 2.58 ms


In [361]:
%%time
### cell 28 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0) & (Broadband_Stats_relative['Change_Download'] < 0.2)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats_relative.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Relative Speed change',
#                   xaxis_title="Country", yaxis_title="Relative Improvement of Speed",
#                   legend_title='Color')
# fig.show()

CPU times: user 748 μs, sys: 80 μs, total: 828 μs
Wall time: 717 μs


In [362]:
cudf_ImprovedCountries_B = cudf.DataFrame()

In [363]:
%%time
## Transfer_pre 29 ##
cudf_ImprovedCountries_B['Change_Upload'] = ImprovedCountries_B['Change_Upload']

CPU times: user 3.52 ms, sys: 377 μs, total: 3.89 ms
Wall time: 2.59 ms


In [364]:
%%time
## Transfer_pre 29 ##
cudf_ImprovedCountries_B['Change_Latency'] = ImprovedCountries_B['Change_Latency']

CPU times: user 4.55 ms, sys: 50 μs, total: 4.6 ms
Wall time: 3.46 ms


In [365]:
%%time
## Transfer_pre 29 ##
cudf_ImprovedCountries_B['Change_Download'] = ImprovedCountries_B['Change_Download']

CPU times: user 3.88 ms, sys: 194 μs, total: 4.07 ms
Wall time: 3.15 ms


In [366]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [367]:
%%time
## Transfer_pre 29 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 4.31 ms, sys: 25 μs, total: 4.33 ms
Wall time: 2.63 ms


In [368]:
%%time
### cell 29 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] < 0)]
# STEFANOS: Disable plotting
# fig = go.Figure()
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=ImprovedCountries_B['Change_Download'],opacity=0.6,name='Broadband'))
# fig.add_trace(go.Bar(x=ImprovedCountries_B.index,y=Mobile_Stats_relative.query("index in @ImprovedCountries_B.index")['Change_Download'],opacity=0.6,name='Mobile'))
# fig.update_layout(barmode='group',
#                   title='Bar Chart of Relative Speed change',
#                   xaxis_title="Country", yaxis_title="Relative Improvement of Speed",
#                   legend_title='Color')
# fig.show()

CPU times: user 640 μs, sys: 68 μs, total: 708 μs
Wall time: 597 μs


In [369]:
cudf_Broadband_Stats_relative = cudf.DataFrame()

In [370]:
%%time
## Transfer_pre 30 ##
cudf_Broadband_Stats_relative['Change_Download'] = Broadband_Stats_relative['Change_Download']

CPU times: user 4.31 ms, sys: 26 μs, total: 4.34 ms
Wall time: 2.8 ms


In [371]:
%%time
### cell 30 ###

Broadband_Stats_relative.sort_values(by=['Change_Download'])

CPU times: user 1.62 ms, sys: 173 μs, total: 1.79 ms
Wall time: 1.04 ms


,Change_Download,Change_Upload,Change_Latency
Name,,,
Marshall Islands,-0.838855,-0.435747,-0.141304
Comoros,-0.726721,-0.973147,0.236025
Guinea-Bissau,-0.492244,-0.403185,0.405405
United States Minor Outlying Islands,-0.406104,-0.920698,1.945148
Saint Helena,-0.399552,-0.162749,-0.015524
...,...,...,...
Nauru,6.308000,23.425648,-0.641304
Tonga,6.369925,5.602930,-0.219388
Wallis and Futuna Islands,7.068718,8.957143,-0.109718


In [372]:
cudf_Mobile_Stats_relative = cudf.DataFrame()

In [373]:
%%time
## Transfer_pre 31 ##
cudf_Mobile_Stats_relative['Change_Download'] = Mobile_Stats_relative['Change_Download']

CPU times: user 2.94 ms, sys: 314 μs, total: 3.26 ms
Wall time: 2.19 ms


In [374]:
%%time
### cell 31 ###

Mobile_Stats_relative.sort_values(by=['Change_Download'])

CPU times: user 614 μs, sys: 0 ns, total: 614 μs
Wall time: 457 μs


,Change_Download,Change_Upload,Change_Latency
Name,,,
Wallis and Futuna Islands,-0.991780,-0.973705,0.237410
Cook Islands,-0.821603,-0.653204,3.692308
Kiribati,-0.801917,-0.697532,1.362869
Cuba,-0.567984,-0.303181,0.176000
Timor-Leste,-0.543723,-0.483994,-0.027027
...,...,...,...
Cayman Islands,3.223961,1.128062,-0.026316
Mauritania,3.328165,4.981119,0.153846
Guinea-Bissau,3.554574,1.512042,-0.583333


In this view we have **Saint Pierre and Miquelon** improving the most relative to its baseline speed and **Wallis and Futuna Islands** losing almost all of its original national average download speed for mobile devices. On the other hand, we have **Antarctica** losing 0.82x its original average speed while Tongo improve its speed by 5x for broadband downloads. Either ends of this view of data would be filled with countries that have low download speeds to begin with, as in that case a minute improvement would also be amplified due to its relative nature.